# LangGraph Components

Building a ReAct agent includes the following steps:
1. User message comes in 
2. Use system prompt with ReAct process to instruct the LLM model
3. Call LLM with the user message to output Thought, Action, Pause
4. Based on the output, a decision has to made to either use an action tool or return using a query loop function 
5. Call tool to get back an observation 
6. Loop observation back into the prompt and send as new message

LangChain Components are:-
1. Prompts
2. Tools 
3. LangGraphs

#### _LangChain Components - Prompts:_

- LangChain has a component called Prompt templates that allow reusable prompts
- prompt does not need f string and can add formatted variables that we can change
- the formatted variables in the reuseable prompt can be changed using user content
- there is langchain hub where alot of prompt templates are available

#### _LangChain Components - Tools:_

- Can get exisiting tools from library
- Tavily tool from langchain community which helps get search results
- langchain community package has many different tools available for use

#### _LangGraph:_

- Biggest part of building an agent is making the loop function
- LangGraph helps to describe and control the flow
- Helps to create cyclic graphs needed for builing agents as it has to have a loop function
- Has built in persistence to remember previous iterations or changes or actions
- enables human in the loop features
- the structure of an agent is depicted as graphs which lead to building langGraphs aimed to support agent and multi-agent flows
- Allows for controlled flows
- Core concepts of langgraph are nodes, edges and conditional edges

Core concepts of LangGraph:
1. Nodes - They are agents or functions
2. Edges - Connect nodes. Eg: call different functions to connect them in the agentic workflow
3. Conditional edges - used when making descisions. Eg: to decide to perform an action or return answer

Example of LangGraph:
- Agent Node - initialises the agent by calling LLM with user message
- Conditional edge - takes result of LLM call and decides to either perform an action or return final answer
- Action function node - one edge connects to a function node that performs an action which calles agent node again
- End Node - Other egde connects to end node

Data/State:
- LangGraph has a state trapped over time
- state is accessible at all parts of the graph - all nodes and edges
- it is local to the graph
- stored in persistence layer which means that the state can be resumed at any point
- intermediate steps are recorded to store previosu actions and observations

### _Build Agent using LangGraph:_

Import langgraph state graph, end node , tools, langchain messages, model, typing and operator:

=> typing and operator is used to construct the agent state

=> use langchain core msgs to differentiate between the type of msgs

    from langgraph.graph import StateGraph, END
    from typing import TypedDict, Annotated
    import operator
    from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
    from langchain_openai import ChatOpenAI
    from langchain_community.tools.tavily_search import TavilySearchResults

Initialise Tavily search tool with max results as 2 or 4:

    tool = TavilySearchResults(max_results=4) #increased number of results

Create agent state with an annotated list of messages to add to over time:

    class AgentState(TypedDict):
        messages: Annotated[list[AnyMessage], operator.add]

Create agent class with 3 functions:

1. Function to call LLM
2. Function to check if action is present
3. Function to perform the action  

Features under init for agent class:
- Takes in model, tools, system prompt as parameters
- Save system prompt message as attribute
- Intialise graph with agent state
- Create a node to call LLM in the graph which takes in a name and function to be executed
- Add another node to the graph for action
- Add conditional edge which connects to the llm node where the edge starts (as the condition is based on the llm output)
- If function returns true, then proceed to the action
- If fucntion returns false, go to end node
- Add edge from action node to llm node (which takes action tool output as observation back to the llm)
- Set entry point for the graph at the llm node
- Compile the graph to run the graph after it is structured and set up 
- Pass in the tools as a dictionary to map the name of the tool to the tool function
- Bind tools to the model so model knows the tools that are avaible for it to call


Create the 3 functions to run:

1. Call lmm function (call_openai) for llm node -
- call the model which takes the state (all nodes and edges will take agent state)
- get list of messages from the state
- add system message to messages list if it is there 
- call the model by invoking 
- return message list
- pass this function to the llm node in the graph 

2. Action fucntion (take_action) for action node -
- Takes in agent state as parameter
- Gte last message in the list
- Need to call tools
- loop through tool calls needed to perform
- invoke that tool from list of tool available and passing the arguments
- append as a tool message

3. Action exists function (exists_action) for conditional edge:

- Take in result after llm is called and take in a boolean as either true or false
- Checks whether we should take action or not
- Take last message from the state
- return true or false if length of the last msg tool calls is greater than 0

    class Agent:

        def __init__(self, model, tools, system=""):
            self.system = system
            graph = StateGraph(AgentState)
            graph.add_node("llm", self.call_openai)
            graph.add_node("action", self.take_action)
            graph.add_conditional_edges(
                "llm",
                self.exists_action,
                {True: "action", False: END}
            )
            graph.add_edge("action", "llm")
            graph.set_entry_point("llm")
            self.graph = graph.compile()
            self.tools = {t.name: t for t in tools}
            self.model = model.bind_tools(tools)

        def exists_action(self, state: AgentState):
            result = state['messages'][-1]
            return len(result.tool_calls) > 0

        def call_openai(self, state: AgentState):
            messages = state['messages']
            if self.system:
                messages = [SystemMessage(content=self.system)] + messages
            message = self.model.invoke(messages)
            return {'messages': [message]}

        def take_action(self, state: AgentState):
            tool_calls = state['messages'][-1].tool_calls
            results = []
            for t in tool_calls:
                print(f"Calling: {t}")
                if not t['name'] in self.tools:      # check for bad tool name from LLM
                    print("\n ....bad tool name....")
                    result = "bad tool name, retry"  # instruct LLM to retry if bad
                else:
                    result = self.tools[t['name']].invoke(t['args'])
                results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
            print("Back to the model!")
            return {'messages': results}

Create system prompt:

EXAMPLE PROMPT -    

    prompt = """You are a smart research assistant. Use the search engine to look up information. \
    You are allowed to make multiple calls (either together or in sequence). \
    Only look up information when you are sure of what you want. \
    If you need to look up some information before asking a follow up question, you are allowed to do that!
    """

Intialise and call agent with model, tool and system prompt:

    model = ChatOpenAI(model="gpt-3.5-turbo")  
    abot = Agent(model, [tool], system=prompt)

Visualise the LangGraph created as image to understand the agent flow:

    from IPython.display import Image

    Image(abot.graph.get_graph().draw_png())

User message and invoke graph with the message:

    messages = [HumanMessage(content="What is the weather in SF and LA?")]
    result = abot.graph.invoke({"messages": messages})

<b>
Agent process displayed after human message is sent to the agent:

\
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_3AAaDSfchif60Drx9QyyOiZS'}

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_Ldw8ZaATItg9oW0SRBlhKtpc'}

Back to the model!

</b>

Last message in result:

    result['messages'][-1].content